In [1]:
# ============================================================
# Loan Default Prediction – Final Feature Engineering Pipeline
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [2]:
# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------
DATA_PATH = "Loan_Default.csv"   # change path if needed
df = pd.read_csv(DATA_PATH)

print("Initial Shape:", df.shape)

Initial Shape: (148670, 34)


In [3]:
# ------------------------------------------------------------
# 2. BASIC CLEANING
# ------------------------------------------------------------

# Remove duplicates
df.drop_duplicates(inplace=True)

In [4]:

# Drop ID-like columns
id_cols = [col for col in df.columns if "id" in col.lower()]
df.drop(columns=id_cols, inplace=True, errors="ignore")

print("After cleaning:", df.shape)

After cleaning: (148670, 33)


In [5]:
# ------------------------------------------------------------
# 3. TARGET VARIABLE
# ------------------------------------------------------------
TARGET = "Status"
X = df.drop(columns=[TARGET])
y = df[TARGET]

In [14]:
# ------------------------------------------------------------
# 4. FEATURE ENGINEERING (DERIVED FEATURES)
# ------------------------------------------------------------

# Financial pressure indicators
X["loan_interest_burden"] = X["loan_amount"] * X["rate_of_interest"]
X["loan_term_pressure"] = X["loan_amount"] / (X["term"] + 1)

# Risk flags
X["high_ltv_flag"] = (X["LTV"] > 80).astype(int)
X["negative_amort_flag"] = (X["Neg_ammortization"] == "Yes").astype(int)
X["business_risk_flag"] = (X["business_or_commercial"] == "Yes").astype(int)

In [13]:
X["Credit_Worthiness"].unique()

array(['l1', 'l2'], dtype=object)

In [17]:
# ------------------------------------------------------------
# 5. SELECT FINAL FEATURES
# ------------------------------------------------------------

numerical_features = [
    "loan_amount",
    "rate_of_interest",
    "term",
    "LTV",
    "Upfront_charges",
    "loan_interest_burden",
    "loan_term_pressure",
]

categorical_features = [
    "Credit_Worthiness",
    "loan_type",
    "Security_Type",
    "loan_purpose",
    "open_credit",
    "business_or_commercial",
    "approv_in_adv",
    "Neg_ammortization"
]

X = X[numerical_features + categorical_features]

In [18]:
# ------------------------------------------------------------
# 6. PREPROCESSING PIPELINES
# ------------------------------------------------------------

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)


In [45]:


# ------------------------------------------------------------
# 7. MODEL PIPELINE
# ------------------------------------------------------------
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier 

lr = LogisticRegression(max_iter=1000)
xgb = XGBClassifier(eval_metric="logloss")
lgbm = LGBMClassifier()

model_pipeline = Pipeline(steps=[("preprocessor", preprocessor),
    ("model", lr)
])


In [46]:
# ------------------------------------------------------------
# 8. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,random_state=42,
    stratify=y
)


In [47]:
# ------------------------------------------------------------
# 9. TRAIN MODEL
# ------------------------------------------------------------

model_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [48]:
# ------------------------------------------------------------
# 10. EVALUATION
# ------------------------------------------------------------

y_pred = model_pipeline.predict(X_test)
print("\nMODEL PERFORMANCE\n")
print(classification_report(y_test, y_pred))

# Predict probability for positive class (class = 1)
y_pred_lr = model_pipeline.predict_proba(X_test)[:, 1]
# Calculate AUC-ROC score
from sklearn.metrics import roc_auc_score
auc_lr = roc_auc_score(y_test, y_pred_lr)
print("Logistic Regression AUC-ROC:", auc_lr)


MODEL PERFORMANCE

              precision    recall  f1-score   support

           0       0.77      0.99      0.86     22406
           1       0.70      0.10      0.17      7328

    accuracy                           0.77     29734
   macro avg       0.73      0.54      0.52     29734
weighted avg       0.75      0.77      0.69     29734

Logistic Regression AUC-ROC: 0.6555125608217854


In [49]:
# ------------------------------------------------------------
# 11. SAVE PIPELINE
# ------------------------------------------------------------

joblib.dump(model_pipeline, "loan_default_model.pkl")
joblib.dump(X.columns.tolist(), "model_features.pkl")

print("\nModel and feature list saved successfully!")


Model and feature list saved successfully!


In [ ]:
# ============================================================
# 12. INFERENCE FUNCTION (RAW USER INPUT)
# ============================================================

def predict_loan_default(raw_input: dict):
    """
    raw_input example:
    {
        "loan_amount": 2500000,
        "rate_of_interest": 9.5,
        "term": 240,
        "LTV": 75,
        "Upfront_charges": 15000,
        "Credit_Worthiness": 720,
        "loan_type": "Home Loan",
        "Security_Type": "Direct",
        "loan_purpose": "Purchase",
        "open_credit": "No",
        "business_or_commercial": "No",
        "approve_in_advance": "Yes",
        "Neg_ammortization": "No",
    }
    """

    model = joblib.load("loan_default_model.pkl")
    feature_cols = joblib.load("model_features.pkl")

    user_df = pd.DataFrame([raw_input])

    # -------- Derive features for inference --------
    user_df["loan_interest_burden"] = user_df["loan_amount"] * user_df["rate_of_interest"]
    user_df["loan_term_pressure"] = user_df["loan_amount"] / (user_df["term"] + 1)
    user_df["high_ltv_flag"] = (user_df["LTV"] > 80).astype(int)
    user_df["negative_amort_flag"] = (user_df["Neg_ammortization"] == "Yes").astype(int)
    user_df["business_risk_flag"] = (user_df["business_or_commercial"] == "Yes").astype(int)
    

    # Add missing columns
    for col in feature_cols:
        if col not in user_df.columns:
            user_df[col] = np.nan

    user_df = user_df[feature_cols]

    prediction = model.predict(user_df)[0]
    probability = model.predict_proba(user_df)[0][1]

    if probability < 0.3:
        risk = "Low Risk"
        action = "Send payment reminder via SMS/Email"
    elif probability < 0.6:
        risk = "Medium Risk"
        action = "Offer flexible EMI or short-term payment plan"
    else:
        risk = "High Risk"
        action = "Assign to recovery agent and initiate call"

    return {
        "default_probability": float(round(probability*100, 2)),
        "risk_level": risk,
        "recommended_action": action
    }

In [54]:
# ------------------------------------------------------------
# 13. SAMPLE INFERENCE TEST
# ------------------------------------------------------------

if __name__ == "__main__":
    sample_input = {
        "loan_amount": 3000000,
        "rate_of_interest": 10.2,
        "term": 240,
        "LTV": 820,
        "Upfront_charges": 1800,
        "Credit_Worthiness": 690,
        "loan_type": "Home Loan",
        "Security_Type": "Direct",
        "loan_purpose": "Purchase",
        "open_credit": "No",
        "business_or_commercial": "No",
        "approve_in_advance": "Yes",
        "Neg_ammortization": "No"
    }
    print("\nInference Output:")
    print(predict_loan_default(sample_input))


Inference Output:
{'default_prediction': 0, 'default_probability': np.float64(0.0)}


In [87]:
df.columns

Index(['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type',
       'loan_purpose', 'Credit_Worthiness', 'open_credit',
       'business_or_commercial', 'loan_amount', 'rate_of_interest',
       'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization',
       'interest_only', 'lump_sum_payment', 'property_value',
       'construction_type', 'occupancy_type', 'Secured_by', 'total_units',
       'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type',
       'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type',
       'Status', 'dtir1'],
      dtype='object')

In [96]:
#df['loan_type'].unique()
#df['Security_Type'].unique()
#df['loan_purpose'].unique()
#df['open_credit'].unique()
#df['business_or_commercial'].unique()
#df['approv_in_adv'].unique()
df['Neg_ammortization'].unique()

array(['not_neg', 'neg_amm', nan], dtype=object)

In [97]:
df['loan_amount'].unique()

array([ 116500,  206500,  406500,  456500,  696500,  706500,  346500,
        266500,  376500,  436500,  136500,  466500,  226500,   76500,
        356500,  156500,  586500,  306500,  316500,  336500,  426500,
        476500,  196500,  186500,  246500,  216500,  506500,  656500,
        256500,  396500,  166500,  236500,  866500,  416500,  386500,
        596500,  606500,   86500,  286500,  146500,  446500,  636500,
        486500,  326500,   56500,  906500,  496500,  106500,  126500,
        296500,  176500, 1376500,  566500,  686500,  556500,  676500,
        366500,  276500,  716500,   66500,  616500,   96500,  826500,
         26500,  666500,  546500,  986500,  526500, 1226500,  726500,
       1486500, 1416500,  536500,  796500,  516500,   46500,  876500,
        576500,  626500, 1506500,  886500,  816500,  646500,  776500,
        746500,  736500,  896500,  836500,  806500, 1386500,  976500,
        926500,  786500,  766500, 1176500, 2006500,  756500, 1136500,
        966500, 1356

In [79]:
pd.DataFrame([sample_input]).columns

Index(['loan_amount', 'rate_of_interest', 'term', 'LTV', 'Upfront_charges',
       'Credit_Worthiness', 'loan_type', 'Security_Type', 'loan_purpose',
       'open_credit', 'business_or_commercial', 'approve_in_advance',
       'Neg_ammortization'],
      dtype='object')

In [84]:
np.array([35.0, 206000.0, 1.0, 0.0, 0.0, 0.5]).reshape(1, -1)

array([[3.50e+01, 2.06e+05, 1.00e+00, 0.00e+00, 0.00e+00, 5.00e-01]])

In [82]:
pd.DataFrame([35.0, 206000.0, 1.0, 0.0, 0.0, 0.5])

,0
0,35.0
1,206000.0
2,1.0
3,0.0
4,0.0
5,0.5


In [85]:
pd.DataFrame(np.array([35.0, 206000.0, 1.0, 0.0, 0.0, 0.5]).reshape(1, -1),columns=['loan_amount', 'rate_of_interest', 'term', 'LTV', 'Upfront_charges',
       'Credit_Worthiness'])

,loan_amount,rate_of_interest,term,LTV,Upfront_charges,Credit_Worthiness
0,35.0,206000.0,1.0,0.0,0.0,0.5


In [55]:
import pandas as pd
import numpy as np
import joblib

# Load trained pipeline
model_pipeline = joblib.load("loan_default_model.pkl")

# Extract components
preprocessor = model_pipeline.named_steps["preprocessor"]
model = model_pipeline.named_steps["model"]

# -----------------------------
# Get feature names
# -----------------------------

# Numerical features
num_features = preprocessor.transformers_[0][2]

# Categorical features (after OneHotEncoding)
cat_transformer = preprocessor.transformers_[1][1]
cat_features = cat_transformer.named_steps["encoder"].get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Combine feature names
all_features = np.concatenate([num_features, cat_features])

# -----------------------------
# Get coefficients
# -----------------------------
coefficients = model.coef_[0]

# Create importance dataframe
feature_importance = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": coefficients,
    "Absolute_Importance": np.abs(coefficients)
})

# Sort by importance
feature_importance = feature_importance.sort_values(
    by="Absolute_Importance", ascending=False
)

print(feature_importance.head(15))

                       Feature  Coefficient  Absolute_Importance
12      Security_Type_Indriect     1.863627             1.863627
13        Security_Type_direct    -1.583911             1.583911
24   Neg_ammortization_neg_amm     0.672679             0.672679
15             loan_purpose_p2     0.580044             0.580044
3                          LTV     0.463433             0.463433
5         loan_interest_burden    -0.418509             0.418509
25   Neg_ammortization_not_neg    -0.392963             0.392963
8         Credit_Worthiness_l2     0.324051             0.324051
4              Upfront_charges    -0.292395             0.292395
0                  loan_amount     0.290954             0.290954
17             loan_purpose_p4    -0.233183             0.233183
10             loan_type_type2     0.221423             0.221423
20  business_or_commercial_b/c     0.221423             0.221423
22         approv_in_adv_nopre     0.219548             0.219548
19             open_credi

In [56]:
import pandas as pd
import numpy as np
import joblib
import statsmodels.api as sm

In [57]:
# Load trained pipeline
pipeline = joblib.load("loan_default_model.pkl")

# Extract preprocessing step
preprocessor = pipeline.named_steps["preprocessor"]

# Load original dataset again
df = pd.read_csv("Loan_Default.csv")

# Target
y = df["Status"]

# Apply SAME feature engineering as training
X = df.drop(columns=["Status"])

X["loan_interest_burden"] = X["loan_amount"] * X["rate_of_interest"]
X["loan_term_pressure"] = X["loan_amount"] / (X["term"] + 1)
X["high_ltv_flag"] = (X["LTV"] > 80).astype(int)
X["negative_amort_flag"] = (X["Neg_ammortization"] == "Yes").astype(int)
X["business_risk_flag"] = (X["business_or_commercial"] == "Yes").astype(int)

# Keep only selected features
feature_cols = joblib.load("model_features.pkl")
X = X[feature_cols]

# Transform features
X_processed = preprocessor.transform(X)

# Add intercept
X_processed = sm.add_constant(X_processed)


In [58]:
# Numerical feature names
num_features = preprocessor.transformers_[0][2]

# Categorical feature names
cat_encoder = preprocessor.transformers_[1][1].named_steps["encoder"]
cat_features = cat_encoder.get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Final feature list
all_features = np.concatenate([["Intercept"], num_features, cat_features])


In [60]:
OneHotEncoder(handle_unknown="ignore", drop="first")


,categories,'auto'
,drop,'first'
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [61]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])


In [62]:
X_processed = preprocessor.transform(X)
X_processed = sm.add_constant(X_processed)

logit_model = sm.Logit(y, X_processed)
result = logit_model.fit(method="lbfgs", maxiter=200)


C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [63]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=1e-5)
X_processed = vt.fit_transform(X_processed)


In [64]:
import numpy as np

print("NaNs:", np.isnan(X_processed).sum())
print("Infs:", np.isinf(X_processed).sum())
print("Rank:", np.linalg.matrix_rank(X_processed))
print("Columns:", X_processed.shape[1])


NaNs: 0
Infs: 0
Rank: 18
Columns: 26


In [65]:
logit_model = sm.Logit(y, X_processed)
result = logit_model.fit(maxiter=100, disp=False)

C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [67]:
print("X_processed shape:", X_processed.shape)
print("Params length:", len(result.params))


X_processed shape: (148670, 26)
Params length: 26


In [68]:
# Numerical feature names
num_features = preprocessor.transformers_[0][2]

# Categorical feature names AFTER drop="first"
cat_encoder = preprocessor.transformers_[1][1].named_steps["encoder"]
cat_features = cat_encoder.get_feature_names_out(
    preprocessor.transformers_[1][2]
)

# Combine feature names
feature_names = np.concatenate([num_features, cat_features])

# Add intercept manually
feature_names = np.insert(feature_names, 0, "Intercept")


In [77]:
summary_df = pd.DataFrame({
    "Feature": feature_names[1:],
    "Coefficient": result.params[1:],
    "P_value": result.pvalues
})

summary_df["Significant_0.05"] = summary_df["P_value"] < 0.05
summary_df["Abs_Coefficient"] = summary_df["Coefficient"].abs()

summary_df = summary_df.sort_values(
    by="Abs_Coefficient", ascending=False
)

print(summary_df.head(15))


                       Feature  Coefficient   P_value  Significant_0.05  \
x13            Upfront_charges    12.389096  0.999926             False   
x14       loan_interest_burden    -9.479734  0.999944             False   
x25            loan_purpose_p4     1.988391       NaN             False   
x9   Neg_ammortization_not_neg     1.645580  0.999978             False   
x22            loan_purpose_p1     1.562609       NaN             False   
x23            loan_purpose_p2     1.533457       NaN             False   
x20     Security_Type_Indriect     1.524815  0.999989             False   
x19            loan_type_type2     1.384544  0.999990             False   
x24            loan_purpose_p3     1.375897       NaN             False   
x11                       term     1.346744  1.000000             False   
x21       Security_Type_direct     1.346744  1.000000             False   
x16       Credit_Worthiness_l1     1.263878       NaN             False   
x8   Neg_ammortization_ne

In [76]:
print(feature_names)
print(result.params)
print(result.pvalues)

['Intercept' 'loan_amount' 'rate_of_interest' 'term' 'LTV'
 'Upfront_charges' 'loan_interest_burden' 'loan_term_pressure'
 'Credit_Worthiness_l1' 'Credit_Worthiness_l2' 'loan_type_type1'
 'loan_type_type2' 'loan_type_type3' 'Security_Type_Indriect'
 'Security_Type_direct' 'loan_purpose_p1' 'loan_purpose_p2'
 'loan_purpose_p3' 'loan_purpose_p4' 'open_credit_nopc' 'open_credit_opc'
 'business_or_commercial_b/c' 'business_or_commercial_nob/c'
 'approv_in_adv_nopre' 'approv_in_adv_pre' 'Neg_ammortization_neg_amm'
 'Neg_ammortization_not_neg']
x1      0.307920
x2     -0.200739
x3      0.001913
x4      0.501742
x5     -0.293344
x6     -0.424961
x7     -0.087587
x8      1.263778
x9      1.645580
x10     0.944760
x11     1.346744
x12     0.617847
x13    12.389096
x14    -9.479734
x15     0.495841
x16     1.263878
x17     0.734694
x18     0.414936
x19     1.384544
x20     1.524815
x21     1.346744
x22     1.562609
x23     1.533457
x24     1.375897
x25     1.988391
x26     0.920957
dtype: float6

In [ ]:
print(feature_names)